
# Deep Learning Models for Two Datasets

This notebook builds and evaluates **two neural-network models** using the datasets from the provided GitHub folder:

1. **Churn_Modelling.csv** → Binary Classification → predict whether a customer will leave.
2. **bengaluru_house_prices.csv** → Regression → predict house price.

The workflow for each dataset is:

**Load Data → EDA → Preprocessing → Train/Test Split → Neural Network → Training → Evaluation → Visualization → Sample Predictions**


In [ ]:

# Install TensorFlow if it is not already installed
# Uncomment the next line in Google Colab if needed:
# !pip install -q tensorflow

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)


## 1. Customer Churn — Binary Classification

In [ ]:

# Load the churn dataset directly from GitHub
churn_url = "https://raw.githubusercontent.com/mahmoudramadan155/ML/main/labs/dl/Churn_Modelling.csv"
churn = pd.read_csv(churn_url)

print("Shape:", churn.shape)
display(churn.head())
print("\nMissing values:")
display(churn.isnull().sum())


In [ ]:

# Basic EDA
print(churn.info())
print("\nTarget distribution:")
display(churn["Exited"].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(data=churn, x="Exited")
plt.title("Customer Churn Distribution")
plt.xlabel("Exited")
plt.ylabel("Number of Customers")
plt.show()


In [ ]:

# Remove columns that are identifiers/text labels rather than useful predictive features
# RowNumber, CustomerId and Surname are not used as model features.
X = churn.drop(columns=["Exited", "RowNumber", "CustomerId", "Surname"])
y = churn["Exited"]

categorical_features = ["Geography", "Gender"]
numeric_features = [col for col in X.columns if col not in categorical_features]

# Preprocessing:
# - Standardize numerical features
# - One-hot encode categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)


In [ ]:

# Build a Feed-Forward Neural Network (ANN)
churn_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_processed.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

churn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)

churn_model.summary()


In [ ]:

# Train the churn model
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history_churn = churn_model.fit(
    X_train_processed,
    y_train,
    validation_split=0.20,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history_churn.history["loss"], label="Train Loss")
axes[0].plot(history_churn.history["val_loss"], label="Validation Loss")
axes[0].set_title("Churn Model Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history_churn.history["accuracy"], label="Train Accuracy")
axes[1].plot(history_churn.history["val_accuracy"], label="Validation Accuracy")
axes[1].set_title("Churn Model Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:

# Evaluate churn model
churn_prob = churn_model.predict(X_test_processed, verbose=0).ravel()
churn_pred = (churn_prob >= 0.50).astype(int)

accuracy = accuracy_score(y_test, churn_pred)
precision = precision_score(y_test, churn_pred, zero_division=0)
recall = recall_score(y_test, churn_pred, zero_division=0)
f1 = f1_score(y_test, churn_pred, zero_division=0)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, churn_pred, zero_division=0))

cm = confusion_matrix(y_test, churn_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Stayed", "Exited"],
    yticklabels=["Stayed", "Exited"]
)
plt.title("Churn Model Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:

# Example predictions
churn_results = X_test.copy()
churn_results["Actual_Exited"] = y_test.values
churn_results["Predicted_Exited"] = churn_pred
churn_results["Probability"] = churn_prob

display(churn_results.head(10))


## 2. Bengaluru House Prices — Regression

In [ ]:

# Load the Bengaluru house-price dataset
house_url = "https://raw.githubusercontent.com/mahmoudramadan155/ML/main/labs/dl/bengaluru_house_prices.csv"
house = pd.read_csv(house_url)

print("Shape:", house.shape)
display(house.head())
print("\nMissing values:")
display(house.isnull().sum())


In [ ]:

# Clean the house-price dataset

house = house.copy()

# Convert "size" such as "2 BHK" into the numeric number of bedrooms
house["bhk"] = (
    house["size"]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

# Convert total_sqft to a numeric value.
# For ranges such as "1200 - 1500", use the midpoint.
def convert_sqft(value):
    try:
        value = str(value).strip()
        if "-" in value:
            low, high = value.split("-", 1)
            return (float(low) + float(high)) / 2
        return float(value)
    except:
        return np.nan

house["total_sqft_num"] = house["total_sqft"].apply(convert_sqft)

# Keep useful columns and remove rows without target/features
house_model = house[
    ["area_type", "location", "total_sqft_num", "bath", "balcony", "bhk", "price"]
].copy()

house_model = house_model.dropna(subset=["location", "total_sqft_num", "bhk", "price"])

# Fill remaining numeric missing values with their median
for col in ["bath", "balcony"]:
    house_model[col] = house_model[col].fillna(house_model[col].median())

# Remove extreme invalid values
house_model = house_model[
    (house_model["total_sqft_num"] > 200) &
    (house_model["price"] > 0) &
    (house_model["bhk"] > 0)
]

print("Cleaned shape:", house_model.shape)
display(house_model.head())


In [ ]:

# Visualize price distribution
plt.figure(figsize=(7, 4))
plt.hist(house_model["price"], bins=50)
plt.title("Bengaluru House Price Distribution")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.show()


In [ ]:

# Prepare features and target
X_house = house_model.drop(columns=["price"])
y_house = house_model["price"]

categorical_house = ["area_type", "location"]
numeric_house = ["total_sqft_num", "bath", "balcony", "bhk"]

house_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_house),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_house)
    ]
)

Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    X_house, y_house,
    test_size=0.20,
    random_state=42
)

Xh_train_processed = house_preprocessor.fit_transform(Xh_train)
Xh_test_processed = house_preprocessor.transform(Xh_test)

# Log-transform the target during training because house prices are strongly right-skewed.
yh_train_log = np.log1p(yh_train)

print("Training shape:", Xh_train_processed.shape)
print("Testing shape:", Xh_test_processed.shape)


In [ ]:

# Build a Feed-Forward Neural Network for regression
house_model_nn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(Xh_train_processed.shape[1],)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.10),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1)
])

house_model_nn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")]
)

house_model_nn.summary()


In [ ]:

# Train the house-price model
early_stop_house = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=True
)

history_house = house_model_nn.fit(
    Xh_train_processed,
    yh_train_log,
    validation_split=0.20,
    epochs=150,
    batch_size=32,
    callbacks=[early_stop_house],
    verbose=1
)


In [ ]:

# Plot house model training history
plt.figure(figsize=(8, 4))
plt.plot(history_house.history["loss"], label="Train Loss")
plt.plot(history_house.history["val_loss"], label="Validation Loss")
plt.title("House Price Model Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.show()


In [ ]:

# Evaluate house-price model in the original price scale
pred_log = house_model_nn.predict(Xh_test_processed, verbose=0).ravel()
house_pred = np.expm1(pred_log)

mae = mean_absolute_error(yh_test, house_pred)
rmse = np.sqrt(mean_squared_error(yh_test, house_pred))
r2 = r2_score(yh_test, house_pred)

print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²  : {r2:.4f}")

# Actual vs Predicted
plt.figure(figsize=(7, 5))
plt.scatter(yh_test, house_pred, alpha=0.5)
line_min = min(yh_test.min(), house_pred.min())
line_max = max(yh_test.max(), house_pred.max())
plt.plot([line_min, line_max], [line_min, line_max], linestyle="--")
plt.title("Actual vs Predicted House Prices")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.show()


In [ ]:

# Example house-price predictions
house_results = Xh_test.copy()
house_results["Actual_Price"] = yh_test.values
house_results["Predicted_Price"] = house_pred

display(house_results.head(10))



## 3. Final Comparison

### Model 1 — Customer Churn
- **Problem type:** Binary classification
- **Neural network output:** Sigmoid
- **Loss:** Binary Cross-Entropy
- **Main metrics:** Accuracy, Precision, Recall, F1-Score
- **Evaluation:** Confusion Matrix

### Model 2 — Bengaluru House Prices
- **Problem type:** Regression
- **Neural network output:** One linear output neuron
- **Loss:** Mean Squared Error
- **Main metrics:** MAE, RMSE, R²
- **Evaluation:** Actual vs Predicted plot

The two models are intentionally built in the same notebook so the preprocessing, neural-network architecture, training, and evaluation steps can be compared easily.


In [ ]:

# Optional: save the trained models
churn_model.save("churn_ann_model.keras")
house_model_nn.save("bengaluru_house_price_ann_model.keras")

print("Models saved successfully.")
